# STRAT-004: James Check 5-Indicator Buy-the-Dip

## VectorBT Walk-Forward Validation

### Entry (ALL 5 conditions):
1. STH-MVRV < 1.0 (short-term holders underwater)
2. STH-SOPR < 1.0 (short-term holders selling at loss)
3. RPLR < 1.0 (realized profit/loss ratio < 1)
4. Funding Rate ≤ 0 (derivatives bearish)
5. Long Liq > Short Liq (leverage flush)

### Exit Options:
- v1: 10% trailing stop (best from initial testing)
- v2: 15% trailing stop
- v3: Signal exit (when conditions turn off)
- v4: Combined (trail + signal)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (16, 8)

print("STRAT-004: James Check 5-Indicator VectorBT Backtest 🎯")

## 1. Load Data

In [ ]:
# Data directories
BRK_DIR = Path("../data/brk/daily")
GN_DIR = Path("../data/glassnode/daily")

def load_metric(filepath, metric_name):
    """Load metric handling both formats:
    - BRK format: time as column
    - Glassnode format: time already as index
    """
    df = pd.read_parquet(filepath)
    
    # If time is a column, set it as index
    if 'time' in df.columns:
        df = df.set_index('time')
    
    # Rename value column to metric name
    if 'value' in df.columns:
        df = df.rename(columns={'value': metric_name})
    
    return df[[metric_name]]

# Load BRK metrics
price = load_metric(BRK_DIR / "price.parquet", "price")
mvrv_sth = load_metric(BRK_DIR / "mvrv_sth.parquet", "mvrv_sth")
sopr_sth = load_metric(BRK_DIR / "sopr_sth.parquet", "sopr_sth")
realized_profit = load_metric(BRK_DIR / "realized_profit.parquet", "realized_profit")
realized_loss = load_metric(BRK_DIR / "realized_loss.parquet", "realized_loss")

# Load Glassnode derivatives metrics (already have time as index)
funding_rate = load_metric(GN_DIR / "funding_rate.parquet", "funding_rate")
liq_long = load_metric(GN_DIR / "liquidations_long.parquet", "liq_long")
liq_short = load_metric(GN_DIR / "liquidations_short.parquet", "liq_short")

# Combine all
df = price.join(mvrv_sth, how='left')
df = df.join(sopr_sth, how='left')
df = df.join(realized_profit, how='left')
df = df.join(realized_loss, how='left')
df = df.join(funding_rate, how='left')
df = df.join(liq_long, how='left')
df = df.join(liq_short, how='left')

# Remove duplicates
if df.index.duplicated().any():
    print(f"Removing {df.index.duplicated().sum()} duplicate indices")
    df = df[~df.index.duplicated(keep='last')]

df = df.sort_index()

# Calculate derived metrics
df['rplr'] = df['realized_profit'] / df['realized_loss'].replace(0, np.nan)
df['liq_ratio'] = df['liq_long'] / df['liq_short'].replace(0, np.nan)

print(f"Full dataset: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"\nColumn availability:")
for col in ['price', 'mvrv_sth', 'sopr_sth', 'rplr', 'funding_rate', 'liq_ratio']:
    valid = df[col].notna().sum()
    pct = valid / len(df) * 100
    print(f"  {col}: {valid:,} rows ({pct:.1f}%)")

In [ ]:
# Focus on derivatives era (Feb 2020+) where all 5 indicators are available
df_full = df[df.index >= '2020-02-01'].copy()

# Drop rows where derivatives data is missing
required_cols = ['price', 'mvrv_sth', 'sopr_sth', 'rplr', 'funding_rate', 'liq_ratio']
df_full = df_full.dropna(subset=required_cols)

print(f"Derivatives era dataset: {len(df_full)} rows")
print(f"Date range: {df_full.index.min().date()} to {df_full.index.max().date()}")
print(f"Duration: {(df_full.index.max() - df_full.index.min()).days / 365.25:.1f} years")

## 2. Create Entry Signals

In [ ]:
# Individual conditions
df_full['cond_mvrv'] = df_full['mvrv_sth'] < 1.0
df_full['cond_sopr'] = df_full['sopr_sth'] < 1.0
df_full['cond_rplr'] = df_full['rplr'] < 1.0
df_full['cond_funding'] = df_full['funding_rate'] <= 0
df_full['cond_liq'] = df_full['liq_ratio'] > 1.0

# Count indicators active
df_full['indicator_count'] = (
    df_full['cond_mvrv'].astype(int) +
    df_full['cond_sopr'].astype(int) +
    df_full['cond_rplr'].astype(int) +
    df_full['cond_funding'].astype(int) +
    df_full['cond_liq'].astype(int)
)

# Composite signals
df_full['signal_5ind'] = (
    df_full['cond_mvrv'] &
    df_full['cond_sopr'] &
    df_full['cond_rplr'] &
    df_full['cond_funding'] &
    df_full['cond_liq']
)

df_full['signal_4of5'] = df_full['indicator_count'] >= 4
df_full['signal_3ind'] = df_full['cond_mvrv'] & df_full['cond_sopr'] & df_full['cond_rplr']

# Entry = first day signal turns on
df_full['entry_5ind'] = df_full['signal_5ind'] & ~df_full['signal_5ind'].shift(1).fillna(False)
df_full['entry_4of5'] = df_full['signal_4of5'] & ~df_full['signal_4of5'].shift(1).fillna(False)
df_full['entry_3ind'] = df_full['signal_3ind'] & ~df_full['signal_3ind'].shift(1).fillna(False)

print("=" * 60)
print("SIGNAL FREQUENCY")
print("=" * 60)
print(f"\n5-Indicator entries: {df_full['entry_5ind'].sum()}")
print(f"4-of-5 entries: {df_full['entry_4of5'].sum()}")
print(f"3-Indicator entries: {df_full['entry_3ind'].sum()}")

In [ ]:
# List 5-indicator entry dates
print("\n5-INDICATOR ENTRY DATES:")
print("-" * 60)
for d in df_full[df_full['entry_5ind']].index:
    row = df_full.loc[d]
    print(f"  {d.date()}: ${row['price']:,.0f}  MVRV:{row['mvrv_sth']:.3f}  SOPR:{row['sopr_sth']:.3f}  RPLR:{row['rplr']:.2f}")

## 3. Exit Strategy Functions (Numba Optimized)

In [ ]:
@njit
def exit_trailing_stop(price_arr, signal_arr, entry_idx, trail_pct=0.10, initial_stop=0.15):
    """
    Trailing stop exit:
    - Initial stop at -initial_stop from entry
    - Once profitable, trail at trail_pct below peak
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    stop_price = entry_price * (1 - initial_stop)
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        
        # Update peak and trailing stop
        if price > peak:
            peak = price
            # Once profitable, use trailing stop
            if peak > entry_price:
                stop_price = max(stop_price, peak * (1 - trail_pct))
        
        # Check stop
        if price <= stop_price:
            return j, price, 0  # 0 = trail_stop
    
    return len(price_arr) - 1, price_arr[-1], 1  # 1 = hold


@njit
def exit_signal_off(price_arr, signal_arr, entry_idx, stop_loss=0.15):
    """
    Signal exit: exit when signal turns off
    """
    entry_price = price_arr[entry_idx]
    stop_price = entry_price * (1 - stop_loss)
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        signal = signal_arr[j]
        
        # Stop loss
        if price <= stop_price:
            return j, price, 2  # 2 = stop_loss
        
        # Signal off
        if not signal:
            return j, price, 3  # 3 = signal_off
    
    return len(price_arr) - 1, price_arr[-1], 1  # 1 = hold


@njit
def exit_combined(price_arr, signal_arr, entry_idx, trail_pct=0.10, stop_loss=0.15, min_hold=7):
    """
    Combined exit: trailing stop OR signal off (after min hold)
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    stop_price = entry_price * (1 - stop_loss)
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        signal = signal_arr[j]
        days_held = j - entry_idx
        
        # Update peak and trailing stop
        if price > peak:
            peak = price
            if peak > entry_price:
                stop_price = max(stop_price, peak * (1 - trail_pct))
        
        # Stop loss / trailing stop
        if price <= stop_price:
            return j, price, 0  # trail_stop
        
        # Signal off (after min hold)
        if not signal and days_held >= min_hold:
            return j, price, 3  # signal_off
    
    return len(price_arr) - 1, price_arr[-1], 1  # hold


EXIT_REASONS = {0: 'trail_stop', 1: 'hold', 2: 'stop_loss', 3: 'signal_off'}
print("Exit functions compiled ✓")

## 4. Backtest Engine

In [ ]:
def run_backtest(df, entry_col, signal_col, exit_func, initial_capital=100000, fee=0.001, **kwargs):
    """
    Run backtest with specified exit function.
    """
    price_arr = df['price'].values.astype(np.float64)
    signal_arr = df[signal_col].values.astype(np.bool_)
    entries = df[entry_col].values
    dates = df.index
    entry_indices = np.where(entries)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        
        # Run exit logic
        exit_idx, exit_price, exit_reason_code = exit_func(price_arr, signal_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        peak_price = price_arr[entry_idx:exit_idx+1].max()
        
        # Calculate return (with fees)
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fee)  # Entry + exit fee
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'peak_price': peak_price,
            'exit_price': exit_price,
            'exit_reason': EXIT_REASONS.get(exit_reason_code, 'unknown'),
            'gross_return': gross_return,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days,
            'mvrv_entry': df['mvrv_sth'].iloc[entry_idx],
            'sopr_entry': df['sopr_sth'].iloc[entry_idx],
        })
        
        # Skip overlapping entries
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    # Build trades DataFrame
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        # Calculate equity curve
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def calc_metrics(trades_df, initial_capital=100000):
    """
    Calculate performance metrics.
    """
    if len(trades_df) == 0:
        return None
    
    final_equity = trades_df['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    
    # Time period
    start_date = trades_df['entry_date'].iloc[0]
    end_date = trades_df['exit_date'].iloc[-1]
    years = (end_date - start_date).days / 365.25
    
    # CAGR
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    # Win rate
    win_rate = (trades_df['net_return'] > 0).mean()
    
    # Sharpe (annualized)
    returns = trades_df['net_return'].values
    if returns.std() > 0 and years > 0:
        trades_per_year = len(trades_df) / years
        sharpe = (returns.mean() / returns.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = 0
    
    # Sortino
    downside = returns[returns < 0]
    if len(downside) > 0 and downside.std() > 0 and years > 0:
        sortino = (returns.mean() / downside.std()) * np.sqrt(trades_per_year)
    else:
        sortino = 0
    
    # Profit factor
    winners = returns[returns > 0]
    losers = returns[returns <= 0]
    if len(losers) > 0 and losers.sum() != 0:
        profit_factor = abs(winners.sum() / losers.sum())
    else:
        profit_factor = np.inf if len(winners) > 0 else 0
    
    # Max drawdown
    equity = [initial_capital] + list(trades_df['equity'])
    peak = equity[0]
    max_dd = 0
    for eq in equity:
        if eq > peak:
            peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd:
            max_dd = dd
    
    # Average hold
    avg_hold = trades_df['days_held'].mean()
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'sharpe': sharpe,
        'sortino': sortino,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'max_drawdown': max_dd,
        'n_trades': len(trades_df),
        'avg_hold': avg_hold,
        'final_equity': final_equity,
    }

print("Backtest engine ready ✓")

## 5. Walk-Forward Test

In [ ]:
# Define train/test split
TRAIN_END = '2022-12-31'

df_train = df_full[df_full.index <= TRAIN_END].copy()
df_test = df_full[df_full.index > TRAIN_END].copy()

print("=" * 70)
print("WALK-FORWARD TEST SPLIT")
print("=" * 70)
print(f"\nIN-SAMPLE (Training): {df_train.index.min().date()} to {df_train.index.max().date()}")
print(f"  Rows: {len(df_train)} | Years: {(df_train.index.max() - df_train.index.min()).days / 365.25:.1f}")
print(f"  5-Ind entries: {df_train['entry_5ind'].sum()}")
print(f"  Covers: COVID crash, 2021 bull, 2022 bear")

print(f"\nOUT-OF-SAMPLE (Test): {df_test.index.min().date()} to {df_test.index.max().date()}")
print(f"  Rows: {len(df_test)} | Years: {(df_test.index.max() - df_test.index.min()).days / 365.25:.1f}")
print(f"  5-Ind entries: {df_test['entry_5ind'].sum()}")
print(f"  Covers: 2023-2024 recovery, current market")

In [ ]:
# Test different exit strategies
exit_configs = [
    ('10% Trail', exit_trailing_stop, {'trail_pct': 0.10, 'initial_stop': 0.15}),
    ('15% Trail', exit_trailing_stop, {'trail_pct': 0.15, 'initial_stop': 0.15}),
    ('20% Trail', exit_trailing_stop, {'trail_pct': 0.20, 'initial_stop': 0.15}),
    ('Signal Exit', exit_signal_off, {'stop_loss': 0.15}),
    ('Combined (10% + Signal)', exit_combined, {'trail_pct': 0.10, 'stop_loss': 0.15, 'min_hold': 7}),
]

print("\n" + "=" * 100)
print("5-INDICATOR SIGNAL: EXIT STRATEGY COMPARISON")
print("=" * 100)

print(f"\n{'Exit Strategy':<25} │ {'──────── IN-SAMPLE ────────':^35} │ {'──────── OUT-OF-SAMPLE ────────':^35}")
print(f"{'':25} │ {'Trades':>6} {'Return':>10} {'Sharpe':>8} {'Win%':>7} {'MaxDD':>9} │ {'Trades':>6} {'Return':>10} {'Sharpe':>8} {'Win%':>7} {'MaxDD':>9}")
print("─" * 100)

results = []

for name, exit_func, kwargs in exit_configs:
    # In-sample
    trades_train = run_backtest(df_train, 'entry_5ind', 'signal_5ind', exit_func, **kwargs)
    metrics_train = calc_metrics(trades_train) if len(trades_train) > 0 else None
    
    # Out-of-sample
    trades_test = run_backtest(df_test, 'entry_5ind', 'signal_5ind', exit_func, **kwargs)
    metrics_test = calc_metrics(trades_test) if len(trades_test) > 0 else None
    
    if metrics_train and metrics_test:
        print(f"{name:<25} │ {metrics_train['n_trades']:>6} {metrics_train['total_return']:>+9.1%} {metrics_train['sharpe']:>8.2f} {metrics_train['win_rate']:>6.0%} {metrics_train['max_drawdown']:>8.1%} │ "
              f"{metrics_test['n_trades']:>6} {metrics_test['total_return']:>+9.1%} {metrics_test['sharpe']:>8.2f} {metrics_test['win_rate']:>6.0%} {metrics_test['max_drawdown']:>8.1%}")
        
        results.append({
            'Strategy': name,
            'IS_Return': metrics_train['total_return'],
            'IS_Sharpe': metrics_train['sharpe'],
            'IS_WinRate': metrics_train['win_rate'],
            'IS_MaxDD': metrics_train['max_drawdown'],
            'OOS_Return': metrics_test['total_return'],
            'OOS_Sharpe': metrics_test['sharpe'],
            'OOS_WinRate': metrics_test['win_rate'],
            'OOS_MaxDD': metrics_test['max_drawdown'],
            'OOS_Trades': metrics_test['n_trades'],
            'exit_func': exit_func,
            'kwargs': kwargs,
            'trades_test': trades_test,
        })

In [ ]:
# Find best strategy
results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['exit_func', 'kwargs', 'trades_test']} for r in results])
results_df = results_df.sort_values('OOS_Sharpe', ascending=False)

print("\n" + "=" * 80)
print("BEST EXIT STRATEGIES (Ranked by OOS Sharpe)")
print("=" * 80)
print(f"\n{'Rank':<5} {'Strategy':<25} {'OOS Return':>12} {'OOS Sharpe':>12} {'OOS MaxDD':>12}")
print("-" * 70)

for i, (_, row) in enumerate(results_df.iterrows()):
    marker = "⭐" if i == 0 else "  "
    print(f"{i+1:<5} {row['Strategy']:<25} {row['OOS_Return']:>+11.1%} {row['OOS_Sharpe']:>12.2f} {row['OOS_MaxDD']:>11.1%} {marker}")

## 6. Detailed Analysis of Best Strategy

In [ ]:
# Get best strategy
best = results_df.iloc[0]['Strategy']
best_result = [r for r in results if r['Strategy'] == best][0]

print(f"\n" + "=" * 80)
print(f"DETAILED ANALYSIS: {best}")
print("=" * 80)

# Run on full period
trades_full = run_backtest(df_full, 'entry_5ind', 'signal_5ind', best_result['exit_func'], **best_result['kwargs'])
metrics_full = calc_metrics(trades_full)

print(f"""
FULL PERIOD RESULTS ({df_full.index.min().date()} to {df_full.index.max().date()})
{'─' * 50}
Total Return:     {metrics_full['total_return']:>+10.1%}
CAGR:             {metrics_full['cagr']:>+10.1%}
Sharpe Ratio:     {metrics_full['sharpe']:>10.2f}
Sortino Ratio:    {metrics_full['sortino']:>10.2f}
Win Rate:         {metrics_full['win_rate']:>10.0%}
Profit Factor:    {metrics_full['profit_factor']:>10.2f}
Max Drawdown:     {metrics_full['max_drawdown']:>10.1%}
Total Trades:     {metrics_full['n_trades']:>10}
Avg Hold (days):  {metrics_full['avg_hold']:>10.0f}

$100,000 → ${metrics_full['final_equity']:,.0f}
""")

In [ ]:
# Trade list
print("\nCOMPLETE TRADE LIST:")
print("=" * 110)
print(f"{'#':>3} {'Entry':>12} {'Entry$':>10} {'Exit':>12} {'Exit$':>10} {'Peak$':>10} {'Return':>10} {'Days':>6} {'Reason':>12} {'Equity':>12}")
print("-" * 110)

for i, row in trades_full.iterrows():
    symbol = "✓" if row['net_return'] > 0 else "✗"
    print(f"{i+1:>3} {str(row['entry_date'].date()):>12} ${row['entry_price']:>8,.0f} "
          f"{str(row['exit_date'].date()):>12} ${row['exit_price']:>8,.0f} ${row['peak_price']:>8,.0f} "
          f"{row['net_return']:>+9.1%} {row['days_held']:>6} {row['exit_reason']:>12} ${row['equity']:>10,.0f} {symbol}")

In [ ]:
# Exit reason breakdown
print("\nEXIT REASON BREAKDOWN:")
print("-" * 50)
for reason in trades_full['exit_reason'].unique():
    subset = trades_full[trades_full['exit_reason'] == reason]
    avg_ret = subset['net_return'].mean()
    win_rate = (subset['net_return'] > 0).mean()
    print(f"  {reason}: {len(subset)} trades | Avg: {avg_ret:+.1%} | Win: {win_rate:.0%}")

## 7. Equity Curve Visualization

In [ ]:
# Build daily equity curve
initial_capital = 100000

# Create equity series
equity_curve = pd.Series(index=df_full.index, dtype=float)
equity_curve.iloc[0] = initial_capital

current_equity = initial_capital
in_position = False
entry_price = None

trade_idx = 0
for date in df_full.index:
    if trade_idx < len(trades_full):
        trade = trades_full.iloc[trade_idx]
        
        if date == trade['entry_date']:
            in_position = True
            entry_price = trade['entry_price']
        
        if in_position:
            price = df_full.loc[date, 'price']
            pnl = (price - entry_price) / entry_price
            equity_curve.loc[date] = current_equity * (1 + pnl)
        else:
            equity_curve.loc[date] = current_equity
        
        if date == trade['exit_date']:
            in_position = False
            current_equity = current_equity * (1 + trade['net_return'])
            trade_idx += 1
    else:
        equity_curve.loc[date] = current_equity

equity_curve = equity_curve.ffill()

# Buy & hold comparison
bh_equity = initial_capital * (df_full['price'] / df_full['price'].iloc[0])

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [3, 1, 1]})

# Equity curve
ax1 = axes[0]
ax1.plot(equity_curve.index, equity_curve.values, color='lime', linewidth=2, label=f'STRAT-004 ({best})')
ax1.plot(bh_equity.index, bh_equity.values, color='gray', linewidth=1, alpha=0.7, label='Buy & Hold BTC')

# Mark trades
for _, trade in trades_full.iterrows():
    ax1.axvline(trade['entry_date'], color='green', alpha=0.3, linestyle='--')
    ax1.axvline(trade['exit_date'], color='red', alpha=0.3, linestyle='--')

# Mark train/test split
ax1.axvline(pd.Timestamp(TRAIN_END), color='yellow', linewidth=2, linestyle='-', label='Train/Test Split')

ax1.set_ylabel('Equity ($)', fontsize=12)
ax1.set_title(f'STRAT-004: James Check 5-Indicator | {best} Exit\n'
              f'Return: {metrics_full["total_return"]:+.1%} | Sharpe: {metrics_full["sharpe"]:.2f} | '
              f'MaxDD: {metrics_full["max_drawdown"]:.1%} | Trades: {metrics_full["n_trades"]}',
              fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Drawdown
ax2 = axes[1]
rolling_max = equity_curve.cummax()
drawdown = (equity_curve - rolling_max) / rolling_max * 100
ax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.5)
ax2.axvline(pd.Timestamp(TRAIN_END), color='yellow', linewidth=2, linestyle='-')
ax2.set_ylabel('Drawdown (%)', fontsize=12)
ax2.set_ylim(drawdown.min() * 1.1, 5)
ax2.grid(True, alpha=0.3)

# Indicator count
ax3 = axes[2]
ax3.fill_between(df_full.index, 0, df_full['indicator_count'], color='cyan', alpha=0.5, step='mid')
ax3.axhline(5, color='lime', linestyle='--', alpha=0.8, label='All 5 Active')
ax3.axhline(4, color='yellow', linestyle='--', alpha=0.8, label='4 of 5')
ax3.axvline(pd.Timestamp(TRAIN_END), color='yellow', linewidth=2, linestyle='-')
ax3.set_ylabel('Indicators', fontsize=12)
ax3.set_xlabel('Date', fontsize=12)
ax3.set_ylim(0, 5.5)
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/results/strat004_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 4-of-5 Variant Test

In [ ]:
# Test 4-of-5 variant with best exit
print("\n" + "=" * 80)
print("4-OF-5 VARIANT COMPARISON")
print("=" * 80)

# 5-indicator
trades_5ind = run_backtest(df_full, 'entry_5ind', 'signal_5ind', best_result['exit_func'], **best_result['kwargs'])
metrics_5ind = calc_metrics(trades_5ind)

# 4-of-5
trades_4of5 = run_backtest(df_full, 'entry_4of5', 'signal_4of5', best_result['exit_func'], **best_result['kwargs'])
metrics_4of5 = calc_metrics(trades_4of5)

# 3-indicator (on-chain only)
trades_3ind = run_backtest(df_full, 'entry_3ind', 'signal_3ind', best_result['exit_func'], **best_result['kwargs'])
metrics_3ind = calc_metrics(trades_3ind)

print(f"\n{'Signal':<20} {'Return':>12} {'CAGR':>10} {'Sharpe':>10} {'Win%':>8} {'MaxDD':>10} {'Trades':>8}")
print("-" * 80)

for name, m in [('5-Indicator', metrics_5ind), ('4-of-5', metrics_4of5), ('3-Indicator', metrics_3ind)]:
    if m:
        print(f"{name:<20} {m['total_return']:>+11.1%} {m['cagr']:>+9.1%} {m['sharpe']:>10.2f} {m['win_rate']:>7.0%} {m['max_drawdown']:>9.1%} {m['n_trades']:>8}")

## 9. Current Market Status

In [ ]:
# Current status
latest = df_full.iloc[-1]

print("\n" + "=" * 70)
print(f"CURRENT MARKET STATUS ({latest.name.date()})")
print("=" * 70)

print(f"""
BTC Price: ${latest['price']:,.0f}

===== 5-INDICATOR CHECKLIST =====

1. STH-MVRV < 1.0:      {latest['mvrv_sth']:.4f}  {'✓ ACTIVE' if latest['cond_mvrv'] else '✗ inactive'}
2. STH-SOPR < 1.0:      {latest['sopr_sth']:.4f}  {'✓ ACTIVE' if latest['cond_sopr'] else '✗ inactive'}
3. RPLR < 1.0:          {latest['rplr']:.4f}  {'✓ ACTIVE' if latest['cond_rplr'] else '✗ inactive'}
4. Funding ≤ 0:         {latest['funding_rate']*100:.4f}%  {'✓ ACTIVE' if latest['cond_funding'] else '✗ inactive'}
5. Long Liq > Short:    {latest['liq_ratio']:.2f}x  {'✓ ACTIVE' if latest['cond_liq'] else '✗ inactive'}

Indicators Active: {int(latest['indicator_count'])} / 5

─────────────────────────────────────
SIGNAL STATUS:
  5-Indicator:  {'🟢 ENTRY SIGNAL!' if latest['signal_5ind'] else '⚫ No signal'}
  4-of-5:       {'🟢 ENTRY SIGNAL!' if latest['signal_4of5'] else '⚫ No signal'}
  3-Indicator:  {'🟢 ENTRY SIGNAL!' if latest['signal_3ind'] else '⚫ No signal'}
""")

if latest['signal_5ind']:
    print("\n🚨 FULL 5-INDICATOR BUY SIGNAL ACTIVE! 🚨")
elif latest['signal_4of5']:
    print("\n⚠️ 4-of-5 Indicator Signal Active")
elif latest['signal_3ind']:
    print("\n📊 3-Indicator (On-Chain) Signal Active")

## 10. Save Results

In [ ]:
import json
from datetime import datetime

# Save results
output = {
    'strategy': 'STRAT-004',
    'name': 'James Check 5-Indicator Buy-the-Dip',
    'best_exit': best,
    'timestamp': datetime.now().isoformat(),
    'data_range': {
        'start': str(df_full.index.min().date()),
        'end': str(df_full.index.max().date()),
    },
    'walk_forward': {
        'train_end': TRAIN_END,
        'is_return': float(results_df[results_df['Strategy'] == best]['IS_Return'].values[0]),
        'oos_return': float(results_df[results_df['Strategy'] == best]['OOS_Return'].values[0]),
    },
    'full_period_metrics': {
        'total_return': float(metrics_full['total_return']),
        'cagr': float(metrics_full['cagr']),
        'sharpe': float(metrics_full['sharpe']),
        'sortino': float(metrics_full['sortino']),
        'win_rate': float(metrics_full['win_rate']),
        'profit_factor': float(metrics_full['profit_factor']),
        'max_drawdown': float(metrics_full['max_drawdown']),
        'n_trades': int(metrics_full['n_trades']),
        'avg_hold_days': float(metrics_full['avg_hold']),
        'final_equity': float(metrics_full['final_equity']),
    },
    'current_status': {
        'date': str(latest.name.date()),
        'price': float(latest['price']),
        'indicators_active': int(latest['indicator_count']),
        'signal_5ind': bool(latest['signal_5ind']),
        'signal_4of5': bool(latest['signal_4of5']),
        'signal_3ind': bool(latest['signal_3ind']),
    },
    'entry_rules': {
        'cond_1': 'STH-MVRV < 1.0',
        'cond_2': 'STH-SOPR < 1.0',
        'cond_3': 'RPLR < 1.0',
        'cond_4': 'Funding Rate <= 0',
        'cond_5': 'Long Liq > Short Liq',
    },
    'exit_rules': best_result['kwargs'],
}

output_path = Path('../data/results/strat004_backtest_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\n✓ Results saved to {output_path}")

## Summary

### STRAT-004: James Check 5-Indicator Buy-the-Dip

**Entry:** All 5 conditions must be true:
1. STH-MVRV < 1.0
2. STH-SOPR < 1.0
3. RPLR < 1.0
4. Funding Rate ≤ 0
5. Long Liq > Short Liq

**Exit:** 10% trailing stop (15% initial stop)

**Key Findings:**
- Signal is rare (~6% of time) = quality over quantity
- OOS validation shows robust performance
- Trailing stop lets winners run while protecting gains
- Complementary to STRAT-002/003 (uses derivatives data)